# Drosophila FTIR — Bulk Inference & FlyWalker Analysis Pipeline

Batch-process multiple raw-PNG folders end-to-end:

| Section | Steps | What it does |
|---------|-------|--------------|
| **FLAGS A** | — | Set video list, model, video set for inference |
| **Section 1** | Phases 1–3 | BG subtraction → videos → inference |
| **FLAGS B** | — | Select videos to open in SLEAP and analyse |
| **Phase 4** | — | Open SLEAP GUIs for selected videos |
| *(manual step)* | — | Correct predictions, save, close SLEAP |
| **Phase 4.4** | — | Trim tail frames *(optional)* |
| **FLAGS C** | — | Re-select subset for analysis *(optional)* |
| **Section 3** | Phases 7–10 | Pre-flight check → convert → FlyWalker |

> **Tip:** Run Section 1 for all videos first, set FLAGS B, run Phase 4 to open SLEAP GUIs,
> correct predictions, then set FLAGS C and run Section 2.
> FLAGS C lets you analyse a different (or smaller) subset than what you opened in SLEAP.


In [88]:
# ============================================================
# FLAGS A — Inference settings
# ============================================================

# List of raw-PNG folder names to process (no file extension):
VIDEO_FOLDER_NAMES_INF = [
    "CantonS_unamp_Fly1.2",
     "CantonS_unamp_Fly3.1",
    # "CantonS_unamp_Fly7.1",
]

INFERENCE_MODEL = "SLEAP"          # "ADPT" or "SLEAP"
VIDEO_SET       = "B"              # "B", "C", or "D"  (video used for inference)
MODEL_NAME      = "drosophila_unet64_setB_260407_092817"  # ADPT: .pt file | SLEAP: folder
#MODEL_NAME     = "best_model.pt"

FPS = 250            # video frame rate (frames per second)
DISTCAL_MAP = {}     # per-video calibration: {video_name: µm/px} — empty = use default 68.7 µm/px

# ── Derived paths helper (do not edit below) ──────────────────────────────
from pathlib import Path
import os, sys

_nb = globals().get('__vsc_ipynb_file__', None)
BASE_DIR = Path(_nb).parent if _nb else Path().resolve()

def _paths(vfn):
    """Return a dict of all derived paths for a given VIDEO_FOLDER_NAME."""
    p = {}
    p["raw"]     = BASE_DIR / "PNGs" / "Raw_PNGs"          / vfn
    p["png_b"]   = BASE_DIR / "PNGs" / "Colormap_PNGs"     / vfn
    p["png_c"]   = BASE_DIR / "PNGs" / "NO_Colormap_PNGs"  / vfn
    p["png_d"]   = BASE_DIR / "PNGs" / "RAW_Ordered_PNGs"  / vfn
    p["vid_b"]   = BASE_DIR / "Videos" / "PROCESSED_Colormap_Videos"    / f"{vfn}.avi"
    p["vid_c"]   = BASE_DIR / "Videos" / "PROCESSED_NO_Colormap_Videos" / f"{vfn}.avi"
    p["vid_d"]   = BASE_DIR / "Videos" / "RAW_Videos"                   / f"{vfn}.avi"
    p["inf_vid"] = {"B": p["vid_b"], "C": p["vid_c"], "D": p["vid_d"]}[VIDEO_SET]
    p["pred_dir"]  = BASE_DIR / "Predictions" / vfn
    p["pred_file"] = p["pred_dir"] / f"{vfn}.predictions.slp"
    p["model"]   = (BASE_DIR / "Models" / "VS Code Models" / MODEL_NAME
                    if INFERENCE_MODEL == "ADPT"
                    else BASE_DIR / "Models" / "SLEAP_Models" / MODEL_NAME)
    p['pred_dir'].mkdir(parents=True, exist_ok=True)
    return p

print(f"Notebook dir  : {BASE_DIR}")
print(f"Inference model: {INFERENCE_MODEL}  |  Video set: {VIDEO_SET}  |  Model: {MODEL_NAME}")
print(f"Videos to process ({len(VIDEO_FOLDER_NAMES_INF)}):")
for _vfn in VIDEO_FOLDER_NAMES_INF:
    print(f"  {_vfn}")


Notebook dir  : c:\Users\pepev\Desktop\103665_THESIS_DL_Model\Model_Design_&_Training\Inference_Pipeline
Inference model: SLEAP  |  Video set: B  |  Model: drosophila_unet64_setB_260407_092817
Videos to process (2):
  CantonS_unamp_Fly1.2
  CantonS_unamp_Fly3.1


---
## Section 1 — Bulk Inference (Phases 1 → 3)

Processes every video in `VIDEO_FOLDER_NAMES_INF`:

- **Phase 1** — Background subtraction: Raw PNGs → Set B (colormap), C (no colormap), D (raw)
- **Phase 2** — PNG sets → lossless AVI videos (FFV1 codec, 250 FPS)
- **Phase 3** — Model inference (ADPT or SLEAP) → `.predictions.slp` per video

After Section 1 completes, **open each `.predictions.slp` file in the SLEAP GUI** to inspect
and correct predictions before running Section 2.


In [89]:
# Phase 1 -- Raw Video Processing (all videos)
import glob, cv2
import numpy as np

CROP            = dict(up=0, down=0, left=0, right=0)
BG_LENGTH       = 10
BG_THRESHOLD    = 50
BG_OFFSET_SIGMA = 2

def _crop(img, c):
    h, w = img.shape[:2]
    r = h - c["down"] if c["down"] else h
    b = w - c["right"] if c["right"] else w
    return img[c["up"]:r, c["left"]:b]

def _build_bg(files, crop, bg_len, threshold):
    sel   = files[-bg_len:] if len(files) >= bg_len else files
    stack = []
    for fp in sel:
        im = cv2.imread(fp)
        if im is not None:
            stack.append(_crop(im.astype(np.float64), crop))
    bg       = np.stack(stack, 0)
    bg_mean  = bg.mean(0)
    bg_std   = bg.std(0)
    dpm      = np.zeros(bg_mean.shape[:2], dtype=np.float64)
    dpm[(bg_mean[:, :, 0] > threshold) & (bg_mean[:, :, 1] > threshold) & (bg_mean[:, :, 2] > threshold)] = 255.0
    return bg_mean, bg_std, dpm

def _filter(img, bg_mean, bg_std, dpm, sigma):
    f = img - bg_mean - sigma * bg_std - dpm[..., None]
    return np.clip(f, 0, 255).astype(np.uint8)

def _hot_colormap(gray):
    g   = cv2.normalize(gray.astype(np.float32), None, 0, 1, cv2.NORM_MINMAX)
    lut = np.vstack([
        np.column_stack([np.linspace(0, 1, 21), np.zeros(21),           np.zeros(21)]),
        np.column_stack([np.ones(22),           np.linspace(0, 1, 22), np.zeros(22)]),
        np.column_stack([np.ones(21),           np.ones(21),           np.linspace(0, 1, 21)]),
    ])
    lut = (lut * 255).astype(np.uint8)
    idx = np.clip((g * (len(lut) - 1)).astype(int), 0, len(lut) - 1)
    return cv2.cvtColor(lut[idx], cv2.COLOR_RGB2BGR)

for VFN in VIDEO_FOLDER_NAMES_INF:
    p = _paths(VFN)
    print(f"\n[Phase 1] {VFN}")
    for d in [p['png_b'], p['png_c'], p['png_d']]:
        d.mkdir(parents=True, exist_ok=True)
    files = sorted(glob.glob(str(p['raw'] / '*.png')))
    if not files:
        print(f"  SKIP — no PNGs in {p['raw']}")
        continue
    print(f"  {len(files)} frames ...")
    bg_mean, bg_std, dpm = _build_bg(files, CROP, BG_LENGTH, BG_THRESHOLD)
    for i, fp in enumerate(files, 1):
        im = cv2.imread(fp)
        if im is None:
            continue
        im_c     = _crop(im.astype(np.float64), CROP)
        filtered = _filter(im_c, bg_mean, bg_std, dpm, BG_OFFSET_SIGMA)
        gray     = cv2.cvtColor(filtered, cv2.COLOR_BGR2GRAY)
        cv2.imwrite(str(p['png_b'] / f'{i}.png'), _hot_colormap(gray))
        cv2.imwrite(str(p['png_c'] / f'{i}.png'), gray)
        cv2.imwrite(str(p['png_d'] / f'{i}.png'), _crop(im, CROP))
        if i % 50 == 0 or i == len(files):
            print(f"    {i}/{len(files)}")
    print(f"  done.")

print("\nPhase 1 complete.")



[Phase 1] CantonS_unamp_Fly1.2
  147 frames ...
    50/147
    100/147
    147/147
  done.

[Phase 1] CantonS_unamp_Fly3.1
  184 frames ...
    50/184
    100/184
    150/184
    184/184
  done.

Phase 1 complete.


In [90]:
# Phase 2 -- Video Export (all videos)

def _make_video(png_dir, out_path, fps, tag):
    pngs = sorted(glob.glob(str(png_dir / '*.png')),
                  key=lambda x: int(Path(x).stem))
    if not pngs:
        print(f"    SKIP — no PNGs in {png_dir.name}"); return
    f0     = cv2.imread(pngs[0])
    h, w   = f0.shape[:2]
    colour = (f0.ndim == 3 and f0.shape[2] == 3)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    wr = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'FFV1'),
                         fps, (w, h), isColor=colour)
    for pp in pngs:
        fr = cv2.imread(pp)
        if fr is None: continue
        if not colour and fr.ndim == 3:
            fr = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)
        wr.write(fr)
    wr.release()
    print(f"    Set {tag}: {out_path.name}  ({len(pngs)} frames, {w}x{h})")

for VFN in VIDEO_FOLDER_NAMES_INF:
    p = _paths(VFN)
    print(f"\n[Phase 2] {VFN}")
    for tag, png_dir, out_vid in [
        ('B', p['png_b'], p['vid_b']),
        ('C', p['png_c'], p['vid_c']),
        ('D', p['png_d'], p['vid_d']),
    ]:
        _make_video(png_dir, out_vid, FPS, tag)

print('\nPhase 2 complete.')



[Phase 2] CantonS_unamp_Fly1.2


    Set B: CantonS_unamp_Fly1.2.avi  (147 frames, 442x138)
    Set C: CantonS_unamp_Fly1.2.avi  (147 frames, 442x138)
    Set D: CantonS_unamp_Fly1.2.avi  (147 frames, 442x138)

[Phase 2] CantonS_unamp_Fly3.1
    Set B: CantonS_unamp_Fly3.1.avi  (184 frames, 370x122)
    Set C: CantonS_unamp_Fly3.1.avi  (184 frames, 370x122)
    Set D: CantonS_unamp_Fly3.1.avi  (184 frames, 370x122)

Phase 2 complete.


In [ ]:
# Phase 3 -- Pose Estimation (all videos)
import subprocess, sys, importlib, cv2, shutil as _shutil
import h5py, json
import numpy as np

sys.path.insert(0, str(BASE_DIR))
import inference as _inf_mod
importlib.reload(_inf_mod)
from inference import sleap_predictions_to_array, write_slp

def _relink_to_setb(pred_file, video_b_path):
    vb = str(Path(video_b_path).resolve())
    with h5py.File(str(pred_file), 'r+') as f:
        raw = f['videos_json'][:]
        new = []
        for v in raw:
            obj = json.loads(v.decode('utf-8') if isinstance(v, (bytes, np.bytes_)) else v)
            obj['filename'] = vb
            if 'backend' in obj:
                obj['backend']['filename'] = vb
            new.append(json.dumps(obj).encode('utf-8'))
        del f['videos_json']
        f.create_dataset('videos_json', data=np.array(new))

for VFN in VIDEO_FOLDER_NAMES_INF:
    p = _paths(VFN)
    print(f"\n[Phase 3] {VFN}")

    # ── Inference ───────────────────────────────────────────────────────
    if INFERENCE_MODEL == 'ADPT':
        cmd = [
            sys.executable, str(BASE_DIR / 'inference.py'),
            '--model',    str(p['model']),
            '--video',    str(p['inf_vid']),
            '--video_b',  str(p['vid_b']),
            '--output',   str(p['pred_file']),
            '--set',      VIDEO_SET,
            '--template', str(BASE_DIR / 'Drosophila_TRAIN_set.slp'),
        ]
        print("  ADPT inference ...")
        r = subprocess.run(cmd, capture_output=False)
        if r.returncode != 0:
            print(f"  ERROR (returncode={r.returncode}) — skipping {VFN}"); continue
        print('  Inference done. Set B already linked.')
    else:
        model_yaml = p['model'] / 'training_config.yaml'
        if model_yaml.exists():
            # sleap-nn (PyTorch) model.
            _sleap_nn = _shutil.which('sleap-nn') or _shutil.which('sleap-nn.exe')
            if _sleap_nn:
                cmd = [
                    _sleap_nn, 'track',
                    '--data_path',   str(p['inf_vid']),
                    '--model_paths', str(p['model']),
                    '--output_path', str(p['pred_file']),
                ]
                print(f"  sleap-nn inference (exe: {_sleap_nn}) ...")
            else:
                # shutil.which('python') can resolve to an unrelated interpreter
                # that lacks sleap_nn -- verify capability before trusting it;
                # fall back to sys.executable, which is guaranteed to have it.
                _python = sys.executable
                _which_python = _shutil.which('python') or _shutil.which('python.exe')
                if _which_python:
                    _check = subprocess.run([_which_python, '-c', 'import sleap_nn'],
                                             capture_output=True, timeout=15)
                    if _check.returncode == 0:
                        _python = _which_python
                cmd = [
                    _python, '-m', 'sleap_nn.cli', 'track',
                    '--data_path',   str(p['inf_vid']),
                    '--model_paths', str(p['model']),
                    '--output_path', str(p['pred_file']),
                ]
                print(f"  sleap-nn inference (python: {_python}) ...")
        else:
            # Old TF SLEAP model
            cmd = ['sleap-track', str(p['inf_vid']), '--model', str(p['model']), '--output', str(p['pred_file'])]
            print("  SLEAP inference ...")
        r = subprocess.run(cmd, shell=False, capture_output=False)
        if r.returncode != 0:
            print(f"  ERROR (returncode={r.returncode}) — skipping {VFN}"); continue

        # ── Relink to Set B ──────────────────────────────────────────────
        _relink_to_setb(p['pred_file'], p['vid_b'])
        print('  Relinked -> Set B.')

        # ── Phase 3.5b: embed node names ──────────────────────────────────
        _cap = cv2.VideoCapture(str(p['vid_b']))
        _n   = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        _cap.release()
        preds = sleap_predictions_to_array(str(p['pred_file']), _n)
        write_slp(
            output_path  = str(p['pred_file']),
            predictions  = preds,
            video_b_path = str(p['vid_b']),
            template_slp = str(BASE_DIR / 'Drosophila_TRAIN_set.slp'),
        )
        print('  Node names embedded.')

    # ── Phase 3.6: Align body axis (thorax = midpoint(head, abdomen)) ──────
    _KP_HEAD=0; _KP_THORAX=1; _KP_ABDOMEN=2
    with h5py.File(str(p['pred_file']), 'r+') as f:
        _pts  = f['points'][:]
        _inst = f['instances'][:]
        _n_aligned = 0
        for _ir in _inst:
            _p0 = int(_ir['point_id_start'])
            _i_h=_p0+_KP_HEAD; _i_t=_p0+_KP_THORAX; _i_a=_p0+_KP_ABDOMEN
            if bool(_pts['visible'][_i_h]) and bool(_pts['visible'][_i_a]):
                _pts['x'][_i_t] = (float(_pts['x'][_i_h]) + float(_pts['x'][_i_a])) / 2.0
                _pts['y'][_i_t] = (float(_pts['y'][_i_h]) + float(_pts['y'][_i_a])) / 2.0
                _n_aligned += 1
        f['points'][...] = _pts
    print(f'  Body axis aligned in {_n_aligned} instances.')

    print(f"  Predictions: {p['pred_file'].name}")

print('\nPhase 3 complete.')

In [97]:
# ============================================================
# FLAGS B — SLEAP correction & analysis selection
# ============================================================
# Select which videos to open in SLEAP (Phase 4) and analyse (Section 2).
# Must be a subset of (or equal to) VIDEO_FOLDER_NAMES_INF.
# Re-run FLAGS C before Section 2 if you want a different subset for analysis.

VIDEO_FOLDER_NAMES_ANA = [
    "CantonS_unamp_Fly1.2",
     "CantonS_unamp_Fly3.1",
    # "CantonS_unamp_Fly7.1",
]

print(f"Videos selected ({len(VIDEO_FOLDER_NAMES_ANA)}):")
for _vfn in VIDEO_FOLDER_NAMES_ANA:
    print(f"  {_vfn}")

Videos selected (2):
  CantonS_unamp_Fly1.2
  CantonS_unamp_Fly3.1


---
## Phase 4 — SLEAP GUI: Open All Videos for Manual Correction

Launches one SLEAP GUI window per video in `VIDEO_FOLDER_NAMES_ANA` simultaneously.
Correct the predictions in each window, then **save (Ctrl+S)** and close.

In [ ]:
# Phase 4 -- Open SLEAP GUIs (all videos)
import shutil, subprocess

# Auto-detect sleap-label from PATH, then fall back to common install locations
_sleap = shutil.which('sleap-label') or shutil.which('sleap-label.exe')
if _sleap is None:
    _conda_roots = [
        Path.home() / 'anaconda3',
        Path.home() / 'miniconda3',
        Path('C:/ProgramData/anaconda3'),
        Path('C:/ProgramData/miniconda3'),
    ]
    _candidates = [Path.home() / '.local' / 'bin' / 'sleap-label.exe']
    for _root in _conda_roots:
        _candidates.append(_root / 'Scripts' / 'sleap-label.exe')
        _envs_dir = _root / 'envs'
        if _envs_dir.exists():
            for _env in _envs_dir.iterdir():
                _candidates.append(_env / 'Scripts' / 'sleap-label.exe')
    for _c in _candidates:
        if _c.exists():
            _sleap = str(_c)
            break

if _sleap is None:
    raise FileNotFoundError(
        'sleap-label not found. '
        'Make sure SLEAP is installed and its Scripts folder is on PATH.'
    )

_opened = []
_missing = []
for VFN in VIDEO_FOLDER_NAMES_ANA:
    p = _paths(VFN)
    if p['pred_file'].exists():
        _pf = str(p['pred_file'])
        # start + empty title launches each as an independent non-blocking window
        # Strip MPLBACKEND=Agg that Streamlit injects — SLEAP needs Qt backend
        import os as _os
        _env = _os.environ.copy()
        _env.pop('MPLBACKEND', None)
        subprocess.Popen([_sleap, _pf], shell=False, env=_env)
        _opened.append(VFN)
    else:
        _missing.append(VFN)

print(f'Opened {len(_opened)} SLEAP GUI(s):')
for _vfn in _opened:
    print(f'  {_vfn}')
if _missing:
    print('Skipped (predictions file not found):')
    for _vfn in _missing:
        print(f'  {_vfn}')
print('Correct and save (Ctrl+S) each video, then Proceed to Phase 5 (Trim) and Section 3.')

---
## Manual Step — SLEAP Correction

**Phase 4 has opened a SLEAP GUI for each video.** Correct the predictions in each window,
then save (**Ctrl+S**) and close, then Proceed to Phase 5 (Trim) and Section 3.

### Correction checklist (per video)

- [ ] **Body keypoints** (head, thorax, abdomen) present in every frame
- [ ] **Leg contacts** look correct — visible when touching glass, absent when in air
- [ ] **False contacts** removed (predicted leg when fly is clearly off ground)
- [ ] **Missed contacts** added (leg clearly touching but not predicted)
- [ ] No obvious tracking drift between consecutive frames
- [ ] Save corrected file (**Ctrl+S** or File → Save)
- *(Optional)* If the fly walks off before the video ends, use **Phase 4.4** below to trim tail frames

### FlyWalker conditions (Phase 4.5 will verify these automatically)

| Condition | Required for |
|-----------|-------------|
| Each leg has ≥ 2 contact events | Any gait graphs |
| Each contact event lasts ≥ 2 frames | Stance/swing detection |
| All 3 legs on each side detected at least once | Left/right gait graphs |
| Head visible in all frames | Body direction computation |
| Abdomen/thorax aligned (head–thorax–abdomen collinear) | Clean gait phase |

> Once all videos are corrected, proceed to **Phase 4.4** (trim if needed),
> then run **FLAGS C** and **Section 2**.


---
## Phase 4.4 — Trim Tail Frames *(optional, per-video)*

If a fly leaves the frame before the video ends, use this cell to clear all predictions
from a given frame onward — without deleting them one by one in SLEAP.

Add an entry to  for each video that needs trimming, using the
**SLEAP frame number** of the first empty frame (1-based, as shown in the SLEAP GUI).
Videos not listed (or with ) are left unchanged.
A backup  is created automatically for each trimmed video.

In [ ]:
# Phase 5 -- Trim Frames (optional, per-video)
# Keep frames in the interval [TRIM_START, TRIM_END] (1-based SLEAP frame numbers).
# TRIM_START = 0 means "keep from the beginning".
# TRIM_END = 0 means "keep until the end".
TRIM_START = {}   # {video_name: first_frame_to_keep}  -- 0 = keep from beginning
TRIM_END = {}     # {video_name: last_frame_to_keep}   -- 0 = keep all frames

import shutil
import sleap_io as sio

_any_trimmed = False
for _vfn in set(list(TRIM_START.keys()) + list(TRIM_END.keys())):
    _start = TRIM_START.get(_vfn, 0) or 0
    _end = TRIM_END.get(_vfn, 0) or 0
    if _start <= 0 and _end <= 0:
        continue   # nothing to trim for this video
    _p = _paths(_vfn)
    if not _p["pred_file"].exists():
        print(f"SKIP {_vfn} -- predictions file not found")
        continue
    _backup = _p["pred_file"].with_suffix(".pre_trim.slp")
    shutil.copy2(str(_p["pred_file"]), str(_backup))
    print(f"{_vfn}: backup saved -> {_backup.name}")

    # Load, keep only frames whose 1-based index lies in [start, end], resave clean.
    # lf.frame_idx is 0-based, so the 1-based index is lf.frame_idx + 1.
    _labels = sio.load_slp(str(_p["pred_file"]))
    _lo = _start if _start > 0 else 1
    _hi = _end if _end > 0 else None
    _keep = [
        lf for lf in _labels
        if (lf.frame_idx + 1) >= _lo and (_hi is None or (lf.frame_idx + 1) <= _hi)
    ]
    _removed = len(_labels) - len(_keep)
    _trimmed = sio.Labels(_keep)
    sio.save_slp(_trimmed, str(_p["pred_file"]))
    _hi_disp = _hi if _hi is not None else 'end'
    print(f"{_vfn}: kept frames [{_lo}, {_hi_disp}] -- removed {_removed} frame(s), {len(_keep)} remain.")
    _any_trimmed = True

if not _any_trimmed:
    print("Phase 5: no videos trimmed (TRIM_START / TRIM_END are empty or all 0).")


In [ ]:
# ============================================================
# FLAGS C — Analysis selection
# ============================================================
# Select which corrected videos to analyse in Section 2.
# Can be the same as FLAGS B or a smaller subset.
# Re-run this cell to change the selection — no need to re-run Sections 1 or Phase 4.

VIDEO_FOLDER_NAMES_ANA = [
    "CantonS_unamp_Fly1.2",
     "CantonS_unamp_Fly3.1",
    # "CantonS_unamp_Fly7.1",
]

print(f"Videos to analyse ({len(VIDEO_FOLDER_NAMES_ANA)}):")
for _vfn in VIDEO_FOLDER_NAMES_ANA:
    print(f"  {_vfn}")

Videos to analyse (2):
  CantonS_unamp_Fly1.3
  CantonS_unamp_Fly1.1


---
## Section 3 — Bulk FlyWalker Analysis (Phases 7 → 10)

Processes every video in `VIDEO_FOLDER_NAMES_ANA`:

- **Phase 4.5** — FlyWalker pre-flight check (verifies all conditions are met)
- **Phase 5** — Convert corrected `.slp` → SLEAP `.analysis.h5`
- **Phase 6** — Extract keypoint tracks → FlyWalker field arrays
- **Phase 7** — Write `TRACKS.mat` (MATLAB struct format)
- **Phase 8** — Run FlyWalker MATLAB analysis → graphs + `ResultSummary.xlsx`

All outputs land in `Predictions/<VIDEO_FOLDER_NAME>/`.


In [101]:
# Phase 6 -- FlyWalker Pre-flight Validation (all videos)
import h5py
import numpy as np

N_KP   = 9
LEG_KP = {'forelegR':3,'forelegL':4,'midlegR':5,'midlegL':6,'hindlegR':7,'hindlegL':8}

# FlyWalker internal parameters (must match Phase 7 TRACKS.mat):
MINFRAME  = 2   # contacts shorter than this (frames) are dropped before analysis
MIN_STEPS = 4   # minimum surviving contact events per leg for step analysis

def _preflight(pred_file, vfn):
    SEP = '=' * 62
    print(SEP)
    print(f'  Pre-flight: {vfn}')
    print(SEP)

    with h5py.File(str(pred_file), 'r') as f:
        pts_raw  = f['points'][:]
        inst_raw = f['instances'][:]
        frm_raw  = f['frames'][:]

    n_frames   = len(frm_raw)
    vis        = np.zeros((n_frames, N_KP), dtype=bool)
    frame_idxs = np.zeros(n_frames, dtype=int)
    for fi, frm in enumerate(frm_raw):
        frame_idxs[fi] = int(frm['frame_idx'])
        i0, i1 = int(frm['instance_id_start']), int(frm['instance_id_end'])
        if i0 >= i1:
            continue
        p0, p1 = int(inst_raw[i0]['point_id_start']), int(inst_raw[i0]['point_id_end'])
        for k in range(min(p1 - p0, N_KP)):
            vis[fi, k] = bool(pts_raw['visible'][p0 + k])

    body_any_vis = vis[:, 0] | vis[:, 1] | vis[:, 2]
    active_mask  = body_any_vis
    n_active     = int(active_mask.sum())
    n_cleared    = n_frames - n_active

    print(f'  Frames: {n_frames}  (active: {n_active}, cleared/empty: {n_cleared})')
    print()

    # -- Check 1: Any leg detected -----------------------------------------------
    any_leg = vis[active_mask, 3:].any()
    tag = 'PASS' if any_leg else 'FAIL'
    print(f'[{tag}] At least 1 leg contact detected in active frames')
    if not any_leg:
        print('      CRITICAL: no legs detected -- FlyWalker will crash immediately.')

    # -- Check 2: Head visible in all active frames ------------------------------
    head_in_active   = vis[:, 0] & active_mask
    head_ok          = bool(head_in_active.sum() == n_active)
    tag = 'PASS' if head_ok else 'WARN'
    missing_head_fis = [fi for fi in range(n_frames) if active_mask[fi] and not vis[fi, 0]]
    print(f'[{tag}] Head visible in all active frames ({head_in_active.sum()}/{n_active} active)')
    if n_cleared:
        cleared_frames = [int(frame_idxs[fi]) + 1 for fi in range(n_frames) if not active_mask[fi]]
        if len(cleared_frames) <= 10:
            print(f'      Cleared frames (intentional): {cleared_frames}')
        else:
            print(f'      Cleared frames: {cleared_frames[0]}..{cleared_frames[-1]} ({n_cleared} frames)')
    if missing_head_fis:
        missing_frames = [int(frame_idxs[fi]) + 1 for fi in missing_head_fis]
        print(f'      Head MISSING: {missing_frames}')
        print( '      Body direction wrong for those frames -> distorted gait phase plots.')

    # -- Check 3: Per-leg contact analysis ---------------------------------------
    print()
    print(f'--- Per-leg contact analysis ---')
    print(f'    (A contact event = one continuous ground-touch. Touch -> lift -> touch = 2 events.)')
    print(f'    FlyWalker ignores contacts shorter than {MINFRAME} frames and needs >= {MIN_STEPS} per leg.')
    leg_contacts = {}
    all_legs_pass = True
    failed_legs = []
    for leg_name, kp_idx in LEG_KP.items():
        contacts = []
        run = 0
        run_start = None
        for fi in range(n_frames):
            if vis[fi, kp_idx]:
                if run == 0:
                    run_start = frame_idxs[fi]
                run += 1
            elif run > 0:
                contacts.append((run, run_start))
                run = 0
        if run > 0:
            contacts.append((run, run_start))
        leg_contacts[leg_name] = contacts

        n_total     = len(contacts)
        short       = [(dur, start) for dur, start in contacts if dur < MINFRAME]
        eff_durs    = [dur for dur, _ in contacts if dur >= MINFRAME]
        n_eff       = len(eff_durs)
        min_eff_dur = min(eff_durs) if eff_durs else 0

        if n_eff >= MIN_STEPS:
            st = 'PASS'
        else:
            st = 'FAIL'
            all_legs_pass = False
            failed_legs.append((leg_name, n_eff))

        if n_total == 0:
            detail = '0 contact events -- NONE DETECTED'
        else:
            detail = f'{n_total} event(s)'
            if short:
                short_frames = [int(s) + 1 for _, s in short]
                detail += f', {len(short)} too short (< {MINFRAME} frames, at frame(s) {short_frames}) -> ignored'
            detail += f' -> {n_eff} usable'
            if n_eff >= MIN_STEPS:
                detail += f'  (shortest: {min_eff_dur} frames)'
            else:
                detail += f'  <-- need {MIN_STEPS - n_eff} more ground-touch event(s) in SLEAP'
        print(f'  [{st}] {leg_name:<10} : {detail}')

    # -- Check 4: Left / Right side completeness ---------------------------------
    print()
    left_ok  = all(vis[active_mask, i].any() for i in [4, 6, 8])
    right_ok = all(vis[active_mask, i].any() for i in [3, 5, 7])
    tag = 'PASS' if left_ok else 'FAIL'
    print(f'[{tag}] Left  side (FL+ML+HL) all detected at least once')
    if not left_ok:
        print( '      Effect: left-side stance/swing graphs will not be produced.')
    tag = 'PASS' if right_ok else 'FAIL'
    print(f'[{tag}] Right side (FR+MR+HR) all detected at least once')
    if not right_ok:
        print( '      Effect: right-side stance/swing graphs will not be produced.')

    # -- Check 5: Tripod gait observed -------------------------------------------
    tripod_ok = ((vis[:,3]&vis[:,6]&vis[:,7]).any() or
                 (vis[:,4]&vis[:,5]&vis[:,8]).any())
    tag = 'PASS' if tripod_ok else 'WARN'
    print(f'[{tag}] Tripod gait observed (FR+ML+HR or FL+MR+HL) at least once')
    if not tripod_ok:
        print('      Effect: tripod-gait section of FlyWalker output will be empty.')

    # -- Check 6: Support polygon crash risk ------------------------------------
    # FlyWalker draws a support polygon per frame and checks where the body axis
    # crosses it (expects 2 crossings). With 1-2 legs the polygon degenerates to
    # a point/line -> only 1 crossing found -> crash.
    # BUT only if that 1/2-leg pattern is in the top-5 most frequent combinations;
    # rare patterns are silently skipped.
    FW_LEG_KP = [4, 6, 8, 3, 5, 7]   # keypoint indices in FlyWalker's leg order
    leg_combos = np.zeros(n_frames, dtype=int)
    for fi in range(n_frames):
        combo = 0
        for j, kp in enumerate(FW_LEG_KP):
            if vis[fi, kp]:
                combo += 2 ** j
        leg_combos[fi] = combo

    combo_counts = np.zeros(64, dtype=int)
    for fi in np.where(active_mask)[0]:
        c = leg_combos[fi]
        if c > 0:
            combo_counts[c] += 1
    top5_combos = set(np.argsort(combo_counts)[::-1][:5].tolist())

    n_legs_per_frame = vis[active_mask, 3:].sum(axis=1)
    active_idxs      = np.where(active_mask)[0]
    bad_mask         = (n_legs_per_frame == 1) | (n_legs_per_frame == 2)
    n_bad            = int(bad_mask.sum())
    n_one            = int((n_legs_per_frame == 1).sum())
    n_two            = int((n_legs_per_frame == 2).sum())

    risky_mask  = np.array([bad_mask[k] and (leg_combos[active_idxs[k]] in top5_combos)
                             for k in range(len(active_idxs))])
    n_risky     = int(risky_mask.sum())

    tag = 'WARN' if n_risky > 0 else ('NOTE' if n_bad > 0 else 'PASS')
    print(f'[{tag}] Support polygon: {n_bad} frame(s) with 1-2 legs '
          f'({n_one} single-leg, {n_two} two-leg), {n_risky} in top-5 -> crash risk')
    if n_risky > 0:
        print(f'      {n_risky} frame(s) use a frequent 1/2-leg pattern -> FlyWalker may crash.')
        print(f'      If it crashes: .fig plots saved, ResultSummary.xlsx NOT written.')
        risky_sleap  = [int(frame_idxs[active_idxs[k]]) + 1
                        for k in np.where(risky_mask)[0]]
        risky_counts = n_legs_per_frame[risky_mask].tolist()
        shown = min(n_risky, 10)
        for sf, nc in zip(risky_sleap[:shown], risky_counts[:shown]):
            print(f'        Frame {sf}: {int(nc)} leg(s) -- check in SLEAP')
        if n_risky > shown:
            print(f'        ... and {n_risky - shown} more.')
        safe_count = n_bad - n_risky
        if safe_count > 0:
            print(f'      {safe_count} remaining frame(s) are rare patterns -- skipped safely.')
    elif n_bad > 0:
        print(f'      All {n_bad} frame(s) are rare patterns -- FlyWalker skips them safely.')

    # -- Check 6b: 5-6 simultaneous contacts ------------------------------------
    many_mask = (n_legs_per_frame >= 5)
    n_many    = int(many_mask.sum())
    if n_many > 0:
        print(f'[NOTE] {n_many} frame(s) with 5-6 legs touching -- FlyWalker warns and skips (no crash).')
        if n_many >= 10:
            print(f'       High count may indicate over-predicted contacts -- check in SLEAP.')

    # -- Check 7: ResultSummary.xlsx -------------------------------------------
    print(f'[NOTE] ResultSummary.xlsx may not be saved on MATLAB R2025b+ (known COM API bug).')
    print( '       All .fig plots ARE saved. Only the Excel summary is affected.')

    # -- Check 8: Single-leg frames (informational) ----------------------------
    # Exactly 1 leg on the ground is biologically unusual and worth inspecting.
    one_leg_ks = [k for k in range(len(active_idxs)) if n_legs_per_frame[k] == 1]
    if one_leg_ks:
        one_sleap = [int(frame_idxs[active_idxs[k]]) + 1 for k in one_leg_ks]
        shown = min(len(one_sleap), 10)
        print(f'[NOTE] {len(one_sleap)} frame(s) with exactly 1 leg touching.')
        print(f'       Frames: {one_sleap[:shown]}{"..." if len(one_sleap) > shown else ""}')
        print( '       Could be real transition or missed label -- check in SLEAP.')

    # -- Check 9: Isolated single-frame contacts --------------------------------
    # A leg contact lasting exactly 1 frame (not at video boundary) is likely
    # noise or a labelling error -- real contacts are usually >= 2 consecutive frames.
    first_frame = int(frame_idxs.min())
    last_frame  = int(frame_idxs.max())
    isolated = []
    for leg_name in LEG_KP:
        for dur, start in leg_contacts[leg_name]:
            sf = int(start)
            if dur == 1 and sf != first_frame and sf != last_frame:
                isolated.append((sf + 1, leg_name))  # 1-based SLEAP frame
    isolated.sort()
    if isolated:
        shown = min(len(isolated), 10)
        print(f'[NOTE] {len(isolated)} isolated single-frame contact(s) (not at video boundary).')
        print( '       Likely noise or missed label -- check in SLEAP.')
        for sf, leg in isolated[:shown]:
            print(f'         Frame {sf}: {leg}')
        if len(isolated) > shown:
            print(f'         ... and {len(isolated) - shown} more.')

    # -- Check 10: Same-side-only frames ----------------------------------------
    # Frames where all leg contacts are on one side only (all-left or all-right)
    # are unusual -- Drosophila gait normally involves both sides simultaneously.
    left_kps  = [4, 6, 8]  # forelegL, midlegL, hindlegL
    right_kps = [3, 5, 7]  # forelegR, midlegR, hindlegR
    same_side = []
    for k in range(len(active_idxs)):
        fi = active_idxs[k]
        left_any  = any(bool(vis[fi, kp]) for kp in left_kps)
        right_any = any(bool(vis[fi, kp]) for kp in right_kps)
        if (left_any or right_any) and (not left_any or not right_any):
            side = 'left' if left_any else 'right'
            same_side.append((int(frame_idxs[fi]) + 1, side))
    if same_side:
        shown = min(len(same_side), 10)
        print(f'[NOTE] {len(same_side)} frame(s) with contacts on one side only.')
        print( '       Could be model error or real gait transition -- check in SLEAP.')
        for sf, side in same_side[:shown]:
            print(f'         Frame {sf}: {side} side only')
        if len(same_side) > shown:
            print(f'         ... and {len(same_side) - shown} more.')

    # -- Summary -----------------------------------------------------------------
    print()
    print(SEP)

    if not any_leg or not all_legs_pass:
        print('  NOT READY -- FlyWalker will crash with no outputs for this video.')
        if failed_legs:
            for leg_name, n_eff in failed_legs:
                needed = MIN_STEPS - n_eff
                print(f'  Cause: {leg_name} has only {n_eff} usable contact event(s) -- need {needed} more.')
            print( '  Fix:   Open the .slp in SLEAP, find frames where the leg touches the ground')
            print( '         but is not labeled, add the missing contacts (>= 2 consecutive frames')
            print( '         each), save, and re-run Phase 4.5.')
    elif n_risky > 0:
        print(f'  LIKELY TO CRASH -- {n_risky} frame(s) with 1-2 legs match a top-5 pattern.')
        print( '  If it crashes: .fig plots saved, ResultSummary.xlsx NOT written.')
        print( '  Check those frames in SLEAP -- may be missed contacts or real transitions.')
    elif n_bad > 0:
        print(f'  READY (low triangle risk) -- {n_bad} frame(s) with 1-2 legs, none in top-5.')
        print( '  FlyWalker will skip those frames silently -- no crash expected.')
    elif not (head_ok and left_ok and right_ok):
        print('  READY with limitations -- some graphs will be missing. See WARN items above.')
    else:
        print('  Proceed to Section 3 (Phase 7).')
    print(SEP)

for VFN in VIDEO_FOLDER_NAMES_ANA:
    p = _paths(VFN)
    if not p['pred_file'].exists():
        print(f'SKIP {VFN} -- predictions file not found: {p["pred_file"]}')
        continue
    _preflight(p['pred_file'], VFN)
    print()


  Pre-flight: CantonS_unamp_Fly1.2
  Frames: 100  (active: 100, cleared/empty: 0)

[PASS] At least 1 leg contact detected in active frames
[PASS] Head visible in all active frames (100/100 active)

--- Per-leg contact analysis ---
    A contact event = one continuous ground-touch (touch->lift->touch = 2 events).
    FlyWalker ignores events < 2 frames; needs >= 4 surviving events per leg.
  [PASS] forelegR   : 7 event(s), 1 too short (< 2 frames, at SLEAP frame(s) [100]) -> ignored -> 6 usable  (shortest: 9 frames)
  [PASS] forelegL   : 7 event(s), 1 too short (< 2 frames, at SLEAP frame(s) [1]) -> ignored -> 6 usable  (shortest: 9 frames)
  [PASS] midlegR    : 8 event(s), 1 too short (< 2 frames, at SLEAP frame(s) [44]) -> ignored -> 7 usable  (shortest: 5 frames)
  [PASS] midlegL    : 5 event(s) -> 5 usable  (shortest: 10 frames)
  [PASS] hindlegR   : 9 event(s), 3 too short (< 2 frames, at SLEAP frame(s) [28, 48, 85]) -> ignored -> 6 usable  (shortest: 3 frames)
  [PASS] hindlegL   

In [ ]:
# Phase 7 -- Export Labels (all videos)
import subprocess, sys, shutil
from pathlib import Path as _P

# Find sleap-convert — check Scripts\ subfolder of the active env first
_scripts = _P(sys.executable).parent / 'Scripts'
_sleap_convert = str(_scripts / 'sleap-convert.exe')
if not _P(_sleap_convert).exists():
    _sleap_convert = shutil.which('sleap-convert') or shutil.which('sleap-convert.exe')
if _sleap_convert is None:
    raise FileNotFoundError('sleap-convert not found. Make sure the active Python environment has sleap-io or sleap-nn installed.')

for VFN in VIDEO_FOLDER_NAMES_ANA:
    p = _paths(VFN)
    print(f"\n[Phase 5] {VFN}")
    ANALYSIS_FILE = p['pred_dir'] / f'{VFN}.analysis.h5'
    cmd = [_sleap_convert, '--format', 'analysis', '--output', str(ANALYSIS_FILE), str(p['pred_file'])]
    r = subprocess.run(cmd, shell=False, capture_output=True, text=True)
    print(r.stdout)
    if r.stderr: print('stderr:', r.stderr)
    if ANALYSIS_FILE.exists():
        print(f"  -> {ANALYSIS_FILE.name}")
    else:
        print(f"  ERROR (returncode={r.returncode}) — analysis.h5 not created")

print('\nPhase 5 complete.')

In [ ]:
# Phase 8 -- Extract Tracks (all videos)
import h5py
import numpy as np
import math

# Store results per video for Phase 7
_phase6_results = {}

def _extract_tracks(analysis_file, fps):
    with h5py.File(str(analysis_file), 'r') as f:
        tracks_raw = f['tracks'][()]
    tracks = np.squeeze(tracks_raw)
    tracks = tracks.transpose(2, 1, 0)
    n_frames, n_kp, _ = tracks.shape

    def _xy(fi, kp):
        x, y = tracks[fi, kp]
        return (None, None) if (np.isnan(x) or np.isnan(y)) else (float(x), float(y))

    def _empty():
        return np.full(n_frames, -1.0, dtype=np.float64)

    time_arr = np.arange(n_frames, dtype=np.float64) / fps
    body_x=_empty(); body_y=_empty()
    dir1=_empty();   dir2=_empty();   dir3=_empty()
    orient=_empty(); stdx=_empty();   stdy=_empty()
    tc_x=_empty();   tc_y=_empty()
    lf_x=_empty();   lf_y=_empty()
    rf_x=_empty();   rf_y=_empty()
    lm_x=_empty();   lm_y=_empty()
    rm_x=_empty();   rm_y=_empty()
    lb_x=_empty();   lb_y=_empty()
    rb_x=_empty();   rb_y=_empty()

    for fi in range(n_frames):
        tx, ty = _xy(fi, 1)
        if tx is not None:
            body_x[fi]=tx; body_y[fi]=ty; tc_x[fi]=tx; tc_y[fi]=ty
        hx, hy = _xy(fi, 0)
        ax, ay = _xy(fi, 2)
        if hx is not None and ax is not None:
            dx = hx-ax; dy = hy-ay
            body_len = math.hypot(dx, dy)
            stdy[fi] = body_len / 2.0; stdx[fi] = 10.0
            if abs(dx) >= abs(dy):
                d3=1.0; m=dy/dx if abs(dx)>1e-9 else 0.0; b=hy-m*hx
                ori=1.0 if (tx is not None and hx>=tx) else 2.0
            else:
                d3=2.0; m=dx/dy if abs(dy)>1e-9 else 0.0; b=hx-m*hy
                ori=1.0 if (ty is not None and hy>=ty) else 2.0
            dir1[fi]=m; dir2[fi]=b; dir3[fi]=d3; orient[fi]=ori
        for arr_x, arr_y, kp in [(lf_x,lf_y,4),(rf_x,rf_y,3),(lm_x,lm_y,6),
                                  (rm_x,rm_y,5),(lb_x,lb_y,8),(rb_x,rb_y,7)]:
            x, y = _xy(fi, kp)
            if x is not None: arr_x[fi]=x; arr_y[fi]=y

    return dict(time_arr=time_arr, body_x=body_x, body_y=body_y,
                dir1=dir1, dir2=dir2, dir3=dir3, orient=orient,
                stdx=stdx, stdy=stdy, tc_x=tc_x, tc_y=tc_y,
                lf_x=lf_x, lf_y=lf_y, rf_x=rf_x, rf_y=rf_y,
                lm_x=lm_x, lm_y=lm_y, rm_x=rm_x, rm_y=rm_y,
                lb_x=lb_x, lb_y=lb_y, rb_x=rb_x, rb_y=rb_y,
                n_frames=n_frames)

for VFN in VIDEO_FOLDER_NAMES_ANA:
    p = _paths(VFN)
    print(f"\n[Phase 6] {VFN}")

    _DISTCAL_DEFAULT_UM_PER_PX = 100 * 1.46 / 2.125  # ≈ 68.7 µm/px for Sets B/C/D
    _um_per_px = (DISTCAL_MAP or {}).get(VFN, _DISTCAL_DEFAULT_UM_PER_PX)
    DISTCAL = 1.0 / _um_per_px  # convert to px/µm for FlyWalker
    print(f"distcal = {DISTCAL:.6f} px/µm  ({_um_per_px:.1f} µm/px for {VFN})")

    ANALYSIS_FILE = p['pred_dir'] / f'{VFN}.analysis.h5'
    if not ANALYSIS_FILE.exists():
        print(f"  SKIP — analysis.h5 not found (run Phase 5 first)"); continue
    res = _extract_tracks(ANALYSIS_FILE, FPS)
    _phase6_results[VFN] = res
    n = res['n_frames']
    def _pct(arr): return f"{(arr != -1).sum()}/{n}"
    print(f"  Frames: {n}")
    print(f"  Body: {_pct(res['body_x'])}  LF:{_pct(res['lf_x'])} RF:{_pct(res['rf_x'])} LM:{_pct(res['lm_x'])} RM:{_pct(res['rm_x'])} LB:{_pct(res['lb_x'])} RB:{_pct(res['rb_x'])}")

print('\nPhase 6 complete.')


In [78]:
# Phase 9 -- Write MATLAB Data (all videos)
import scipy.io as sio
import numpy as np
if "_phase6_results" not in globals(): _phase6_results = {}

def row(arr):
    return np.asarray(arr, dtype=np.float64).reshape(1, -1)

for VFN in VIDEO_FOLDER_NAMES_ANA:
    p = _paths(VFN)
    print(f"\n[Phase 7] {VFN}")
    if VFN not in _phase6_results:
        print("  SKIP — Phase 6 results not found (run Phase 6 first)"); continue
    r = _phase6_results[VFN]
    TRACKS_MAT = p['pred_dir'] / f'{VFN}.TRACKS.mat'
    v_dict = {
        'time':                    row(r['time_arr']),
        'CurrentBodyX':            row(r['body_x']),
        'CurrentBodyY':            row(r['body_y']),
        'CurrentBodyDirection1':   row(r['dir1']),
        'CurrentBodyDirection2':   row(r['dir2']),
        'CurrentBodyDirection3':   row(r['dir3']),
        'CurrentBodyOrientation':  row(r['orient']),
        'CurrentBodyStdX':         row(r['stdx']),
        'CurrentBodyStdY':         row(r['stdy']),
        'CurrentLeftFrontLegX':    row(r['lf_x']),
        'CurrentLeftFrontLegY':    row(r['lf_y']),
        'CurrentRightFrontLegX':   row(r['rf_x']),
        'CurrentRightFrontLegY':   row(r['rf_y']),
        'CurrentLeftMiddleLegX':   row(r['lm_x']),
        'CurrentLeftMiddleLegY':   row(r['lm_y']),
        'CurrentRightMiddleLegX':  row(r['rm_x']),
        'CurrentRightMiddleLegY':  row(r['rm_y']),
        'CurrentLeftBackLegX':     row(r['lb_x']),
        'CurrentLeftBackLegY':     row(r['lb_y']),
        'CurrentRightBackLegX':    row(r['rb_x']),
        'CurrentRightBackLegY':    row(r['rb_y']),
        'TC_x':                    row(r['tc_x']),
        'TC_y':                    row(r['tc_y']),
    }
    p_dict = {
        'fps':                     np.array([[float(FPS)]]),
        'distcal':                 np.array([[DISTCAL]]),
        'minframe':                np.array([[2.0]]),  # matches p.minframe in Parameters.m
        'fixed_body_length':       np.array([[0.0]]),
        'fixed_body_length_value': np.array([[0.0]]),
        'CenterFromFront':         np.array([[0.0]]),
        'cut': {'up':np.array([[0.0]]),'down':np.array([[0.0]]),'left':np.array([[0.0]]),'right':np.array([[0.0]])},
    }
    sio.savemat(str(TRACKS_MAT), {'v': v_dict, 'p': p_dict})
    print(f"  -> {TRACKS_MAT.name}")

print('\nPhase 7 complete.')



[Phase 7] CantonS_unamp_Fly1.3
  -> CantonS_unamp_Fly1.3.TRACKS.mat

[Phase 7] CantonS_unamp_Fly1.1
  -> CantonS_unamp_Fly1.1.TRACKS.mat

Phase 7 complete.


In [79]:
# Phase 10 -- FlyWalker Analysis (all videos)
import shutil, subprocess, glob as _glob

# Set MATLAB_EXE manually if MATLAB is not on PATH, e.g.:
# MATLAB_EXE = r'C:\Program Files\MATLAB\R2025b\bin\matlab.exe'
MATLAB_EXE = None

if MATLAB_EXE is None:
    MATLAB_EXE = shutil.which('matlab') or shutil.which('matlab.exe')
if MATLAB_EXE is None:
    _cands = sorted(_glob.glob(r'C:\Program Files\MATLAB\*\bin\matlab.exe'), reverse=True)
    if _cands: MATLAB_EXE = _cands[0]
if MATLAB_EXE is None:
    raise FileNotFoundError('MATLAB not found. Set MATLAB_EXE manually above.')
print('MATLAB : ' + str(MATLAB_EXE))

FLYWALKER_DIR = BASE_DIR / 'FlyWalker_Results_Gen'
fw_dir = str(FLYWALKER_DIR).replace('\\', '/')

for VFN in VIDEO_FOLDER_NAMES_ANA:
    p = _paths(VFN)
    print(f"\n[Phase 8] {VFN}")
    TRACKS_MAT = p['pred_dir'] / f'{VFN}.TRACKS.mat'
    if not TRACKS_MAT.exists():
        print("  SKIP — TRACKS.mat not found (run Phase 7 first)"); continue
    mat_path = str(TRACKS_MAT).replace('\\', '/')
    MATLAB_SCRIPT = p['pred_dir'] / 'run_flywalker.m'
    with open(str(MATLAB_SCRIPT), 'w') as _fh:
        _fh.write("addpath('" + fw_dir + "');\n")
        _fh.write("try\n")
        _fh.write("    EvaluateFlyTable_activex_hexa('" + mat_path + "', []);\n")
        _fh.write("catch e\n")
        _fh.write("    disp('ERROR:'); disp(e.message);\n")
        _fh.write("end\n")
        _fh.write("exit;\n")
    script_path = str(MATLAB_SCRIPT).replace('\\', '/')
    run_cmd = "run('" + script_path + "')"
    success = False
    for flags in [
        [MATLAB_EXE, '-nosplash', '-nodesktop', '-wait', '-r', run_cmd],
        [MATLAB_EXE, '-nosplash',               '-wait', '-r', run_cmd],
    ]:
        result = subprocess.run(flags, capture_output=True, text=True, timeout=1800)
        if result.returncode == 0:
            success = True; break
    if not success:
        print('  Headless MATLAB failed. Run manually:')
        print("  run('" + script_path + "')")
    else:
        print(f"  Done. Outputs in: {p['pred_dir'].name}/")

print('\nPhase 8 complete.')


MATLAB : C:\Program Files\MATLAB\R2025b\bin\matlab.EXE

[Phase 8] CantonS_unamp_Fly1.3
  Done. Outputs in: CantonS_unamp_Fly1.3/

[Phase 8] CantonS_unamp_Fly1.1
  Done. Outputs in: CantonS_unamp_Fly1.1/

Phase 8 complete.


---
## Done!

All outputs are saved per-video in `Predictions/<VIDEO_FOLDER_NAME>/`:

| File | Contents |
|------|----------|
| `<name>.predictions.slp` | Corrected SLEAP keypoint predictions |
| `<name>.analysis.h5` | SLEAP analysis HDF5 (tracks array) |
| `<name>.TRACKS.mat` | FlyWalker-compatible MATLAB struct |
| `run_flywalker.m` | MATLAB script for manual re-run |
| `ResultSummary.xlsx` | FlyWalker gait analysis results |
| `*.png` | FlyWalker gait graphs |
